# Chapter 05-11 · Gradient boosting

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** the loop is eight lines; knowing when to
stop it is the chapter

**Prerequisites:** 05-10 for trees, 05-08 for bias and variance, 05-06 for gradient descent, 05-05 for
residuals.

**Position in the learning path:** module 05, chapter 11 of 12.

---

## Why this matters

A random forest fits many trees **independently** to the same problem and averages them. Each tree tries
to solve the whole task and the vote cancels their disagreements.

**Boosting fits them in sequence, each one to the errors the previous ones left.** No tree tries to solve
the whole task; each is handed what remains. The trees cooperate rather than vote, and the ensemble
corrects its own mistakes.

That single change turns the same weak building block into the model that wins most competitions on
tabular data - and it introduces a failure mode the forest did not have. **05-10 established that more
trees can never hurt a forest. For a boosted model, more trees is exactly how you ruin it**, and this
chapter measures the point at which that happens.

**The name is not decoration.** Fitting a tree to the residuals is a step of gradient descent - 05-06's
loop - taken in the space of *functions* rather than the space of coefficients. The learning rate is the
same learning rate.

## What you will be able to do

- Write the boosting loop from scratch and match scikit-learn to four decimal places
- Explain why fitting to the residuals is a gradient step
- Read the two curves and find the round at which to stop
- Say what the learning rate trades against, with measured numbers
- Choose between a forest and a boosted model for a given problem, and say why

## Warm-up: retrieve, do not reread

1. In 05-10, what happens to a forest's held-out error as trees are added?
2. In 05-06, what does the learning rate control, and what happens when it is too large?
3. In 05-05, what is a residual, and why does its sign matter?

<br>

*Answers: (1) it falls and flattens - never worse, only slower. (2) the step size; too large and the loop
oscillates or diverges. (3) `actual - prediction`; the sign carries the structure that every metric
throws away.*

## The loop, written out

Start with a constant - the mean. Then repeat:

1. **Compute the residuals** of what you have so far.
2. **Fit a small tree to those residuals**, not to the target.
3. **Add a fraction of its predictions** to the running model.

The fraction is the learning rate. That is the whole algorithm.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import (GradientBoostingRegressor, HistGradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor


# SYNTHETIC: 05-10's curve. TRUTH: 3 sin(1.2x) + 0.5x, noise sd 1.5
def true_curve(x):
    return np.sin(1.2 * x) * 3 + 0.5 * x


NOISE_SD = 1.5
curve_rng = np.random.default_rng(7)
curve_x = curve_rng.uniform(-4, 4, 400)
curve_y = true_curve(curve_x) + curve_rng.normal(0, NOISE_SD, 400)
fit_x, held_x, fit_y, held_y = train_test_split(curve_x, curve_y, test_size=0.5,
                                                random_state=0)

LEARNING_RATE = 0.3
ROUNDS = 60

running_fit = np.full(len(fit_y), fit_y.mean())        # start at the constant
running_held = np.full(len(held_y), fit_y.mean())
history = [{"round": 0,
            "train RMSE": float(np.sqrt(((fit_y - running_fit) ** 2).mean())),
            "held-out RMSE": float(np.sqrt(((held_y - running_held) ** 2).mean()))}]

for step in range(1, ROUNDS + 1):
    residual = fit_y - running_fit                      # what is still wrong
    stump = DecisionTreeRegressor(max_depth=2, random_state=0).fit(
        fit_x.reshape(-1, 1), residual)                 # fit a tree TO THE ERRORS
    running_fit = running_fit + LEARNING_RATE * stump.predict(fit_x.reshape(-1, 1))
    running_held = running_held + LEARNING_RATE * stump.predict(held_x.reshape(-1, 1))
    history.append({"round": step,
                    "train RMSE": float(np.sqrt(((fit_y - running_fit) ** 2).mean())),
                    "held-out RMSE": float(np.sqrt(((held_y - running_held) ** 2).mean()))})

history = pd.DataFrame(history)
print(history[history["round"].isin([0, 1, 2, 3, 5, 10, 20, 40, 60])]
      .to_string(index=False, float_format=lambda v: "%.4f" % v))

library = GradientBoostingRegressor(n_estimators=ROUNDS, learning_rate=LEARNING_RATE,
                                    max_depth=2, random_state=0).fit(
    fit_x.reshape(-1, 1), fit_y)
print("\nscikit-learn, same settings : held-out RMSE %.4f"
      % np.sqrt(((held_y - library.predict(held_x.reshape(-1, 1))) ** 2).mean()))
print("the loop above              : held-out RMSE %.4f" % history["held-out RMSE"].iloc[-1])

**Eight lines reproduce `GradientBoostingRegressor` to four decimal places - 1.6097 both.**

**And the two columns already show the problem.** The held-out error falls to **1.5179 at round 10** and
then *rises*, reaching 1.6097 by round 60, while the training error keeps falling from 1.4192 to 1.0731.

> **05-10's rule that "more trees is never worse" does not survive here.** It was true of a forest because
> every tree estimated the same quantity and averaging cannot hurt. In boosting each tree is fitted to
> what the previous ones left, so once the real signal is gone the trees start fitting the noise - and
> adding more of them makes the model worse, permanently.

**The number of rounds is a capacity dial**, exactly like polynomial degree in 05-07 and depth in 05-10,
and it has an optimum you can overshoot.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6))

left.plot(history["round"], history["train RMSE"], color="#0072B2", linewidth=2.4,
          label="on the fitting rows")
left.plot(history["round"], history["held-out RMSE"], color="#D55E00", linewidth=2.4,
          label="on held-out rows")
best_round = int(history.loc[history["held-out RMSE"].idxmin(), "round"])
left.axvline(best_round, color="#009E73", linewidth=2,
             label="best at round %d (%.4f)" % (best_round,
                                                history["held-out RMSE"].min()))
left.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.6,
             label="noise floor, %.2f" % NOISE_SD)
left.set_xlabel("boosting round")
left.set_ylabel("RMSE")
left.set_title("Held-out error turns around. A forest's never does.", fontsize=11)
left.legend(fontsize=8.5)

grid = np.linspace(-4, 4, 400)
for rounds, colour in [(1, "#CCCCCC"), (3, "#88BBDD"), (10, "#009E73"), (60, "#D55E00")]:
    staged = GradientBoostingRegressor(n_estimators=rounds, learning_rate=LEARNING_RATE,
                                       max_depth=2, random_state=0).fit(
        fit_x.reshape(-1, 1), fit_y)
    right.plot(grid, staged.predict(grid.reshape(-1, 1)), color=colour, linewidth=2.2,
               label="after %d rounds" % rounds)
right.plot(grid, true_curve(grid), color="#000000", linewidth=2.4, linestyle="--",
           label="the truth")
right.scatter(fit_x, fit_y, s=12, alpha=0.25, color="#999999")
right.set_xlabel("x")
right.set_ylabel("y")
right.set_title("The model assembling itself, one correction at a time", fontsize=11)
right.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

**The right panel is the algorithm working.** After one round the model is a constant plus one small
correction - four steps. By round 10 it tracks the curve. By round 60 it has grown wiggles that are not
in the truth, and the left panel prices them.

## Why fitting the residuals is a gradient step

The name "gradient boosting" is precise, and the connection to 05-06 is worth making explicit because it
explains everything else in the chapter.

For squared-error loss on a single row, with `F` the current model:

$$L = \tfrac{1}{2}\left(y - F(x)\right)^2 \qquad\Longrightarrow\qquad
\frac{\partial L}{\partial F(x)} = -\left(y - F(x)\right) = -\,\text{residual}$$

**The negative gradient of the loss with respect to the prediction *is* the residual.** So "fit a tree to
the residuals and add a fraction of it" is:

$$F \leftarrow F - \eta \cdot \left(\text{an approximation of } \frac{\partial L}{\partial F}\right)$$

which is 05-06's update rule with one change: **the step is taken in the space of functions, not the
space of coefficients.** Gradient descent nudges `w`; boosting adds a whole tree.

**Three things follow immediately, and they are the practical content of this chapter.**

**The learning rate is the same learning rate.** Too large and the model overshoots; too small and it
crawls. 05-06's trade-off applies unchanged.

**Any differentiable loss works.** Replace squared error with absolute error and the tree is fitted to
the *sign* of the residual instead - which is what `loss="absolute_error"` does, and why boosting can be
made robust to outliers when 05-04's MSE could not be.

**The gradient is *approximated* by a tree, not computed exactly.** A depth-2 tree cannot represent an
arbitrary residual pattern; it captures the biggest piece and leaves the rest for the next round. That is
what makes the ensemble grow rather than converge in one step.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharey=True)

for ax, rounds in zip(axes, [0, 1, 5, 20]):
    if rounds == 0:
        residual = fit_y - fit_y.mean()
    else:
        model = GradientBoostingRegressor(n_estimators=rounds, learning_rate=LEARNING_RATE,
                                          max_depth=2, random_state=0).fit(
            fit_x.reshape(-1, 1), fit_y)
        residual = fit_y - model.predict(fit_x.reshape(-1, 1))
    ax.scatter(fit_x, residual, s=16, alpha=0.55, color="#0072B2")
    ax.axhline(0, color="#000000", linewidth=1.4)
    ax.set_xlabel("x")
    ax.set_title("after %d round%s\nresidual sd %.2f"
                 % (rounds, "" if rounds == 1 else "s", residual.std()), fontsize=10.5)

axes[0].set_ylim(-8, 8)
axes[0].set_ylabel("residual - what the next tree is fitted to")
plt.tight_layout()
plt.show()

**Each panel is the target of the next tree.**

At round 0 the residuals are the whole signal - the sine wave is plainly visible, and any tree can find
something in it. After one round the largest piece is gone. By round 20 the structure has been extracted
and what remains is a cloud with a standard deviation of **1.34**.

**That 1.34 is *below* the 1.5 the data was built with, and the discrepancy is the whole warning.** These
are the rows the model was fitted on, so part of their noise has already been absorbed - which is 05-07's
point that training error can go below the noise floor while held-out error cannot. The held-out RMSE at
round 20 was 1.5371, on the wrong side of 1.50 and already climbing.

**That is why the ensemble stops helping and starts hurting.** Once the residual plot is a cloud - 05-05's
signature for a finished model - the next tree can only fit noise, and boosting will fit it enthusiastically
because nothing tells it to stop.

## The learning rate, and what it actually buys

### Predict before running

Below, the same model is fitted at six learning rates from 1.0 down to 0.01, each for 500 rounds. Commit
to an answer: which learning rate reaches the **lowest** held-out error?

In [ ]:
rate_rows = []
for rate in [1.0, 0.5, 0.3, 0.1, 0.05, 0.01]:
    model = GradientBoostingRegressor(n_estimators=500, learning_rate=rate, max_depth=2,
                                      random_state=0).fit(fit_x.reshape(-1, 1), fit_y)
    per_round = np.array([float(np.sqrt(((held_y - prediction) ** 2).mean()))
                          for prediction in model.staged_predict(held_x.reshape(-1, 1))])
    best = int(np.argmin(per_round))
    rate_rows.append({"learning rate": rate, "best round": best + 1,
                      "best held-out RMSE": per_round[best],
                      "RMSE at round 500": per_round[-1],
                      "damage from overshooting": per_round[-1] - per_round[best]})
print(pd.DataFrame(rate_rows).to_string(index=False, float_format=lambda v: "%.4f" % v))
print("\nthe noise floor is %.2f" % NOISE_SD)

**They all reach essentially the same place - 1.5166 to 1.5472 - and they differ completely in how they
get there and what happens if you keep going.**

| learning rate | best round | best RMSE | RMSE at round 500 |
|---|---|---|---|
| 1.00 | 8 | 1.6099 | **2.0709** |
| 0.30 | 10 | 1.5179 | 2.0143 |
| 0.10 | 33 | **1.5166** | 1.7820 |
| 0.01 | 323 | 1.5199 | **1.5232** |

> **The learning rate does not change how good the model can get. It changes how many rounds you need,
> and how much it costs you to get the number of rounds wrong.**

At `rate = 1.0` the optimum is at round 8 and by round 500 the model is **0.46 RMSE worse** than its own
best. At `rate = 0.01` the optimum is at round 323 and running to 500 costs **0.0033**.

**Which is the whole argument for the standard recipe:** a small learning rate, a generous number of
rounds, and **early stopping to pick the round**. The small rate makes the curve flat near its minimum,
so stopping in roughly the right place is good enough - and "roughly the right place" is all a validation
set can identify.

**The cost is compute.** 323 rounds at rate 0.01 against 10 at rate 0.3 is thirty times the trees for
0.0013 of RMSE. On this problem that is a bad trade; on a problem where 0.001 matters it is the reason
production models run to thousands of rounds.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8))

for rate, colour in [(1.0, "#D55E00"), (0.3, "#E69F00"), (0.1, "#0072B2"),
                     (0.01, "#009E73")]:
    model = GradientBoostingRegressor(n_estimators=500, learning_rate=rate, max_depth=2,
                                      random_state=0).fit(fit_x.reshape(-1, 1), fit_y)
    per_round = np.array([float(np.sqrt(((held_y - prediction) ** 2).mean()))
                          for prediction in model.staged_predict(held_x.reshape(-1, 1))])
    ax.plot(np.arange(1, 501), per_round, color=colour, linewidth=2.2,
            label="rate %.2f" % rate)
    best = int(np.argmin(per_round))
    ax.plot([best + 1], [per_round[best]], "o", color=colour, markersize=9)

ax.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.6,
           label="noise floor, %.2f" % NOISE_SD)
ax.set_xscale("log")
ax.set_ylim(1.45, 2.3)
ax.set_xlabel("boosting round (log scale)")
ax.set_ylabel("held-out RMSE")
ax.set_title("Every rate reaches the same floor. Only the shape of the climb differs.",
             fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The dots are each rate's own optimum, and they sit at nearly the same height.** What differs is how far
left they are and how steeply the curve climbs afterwards - the orange line is nearly vertical past its
minimum, the green one is almost flat.

**That flatness is what you are buying with a small learning rate: forgiveness.**

## Early stopping: letting the model find its own round

Reading the best round off a plot works, and it is not what you do in practice. **Hold out a slice, watch
the error on it, and stop when it has not improved for a while** - the library does this for you.

In [ ]:
stopping_rows = []
for rate in [0.10, 0.05]:
    model = HistGradientBoostingRegressor(
        learning_rate=rate, max_iter=1000, early_stopping=True,
        validation_fraction=0.2, n_iter_no_change=20, random_state=0).fit(
        fit_x.reshape(-1, 1), fit_y)
    stopping_rows.append({
        "learning rate": rate, "early stopping": "yes",
        "iterations used": int(model.n_iter_),
        "held-out RMSE": float(np.sqrt(((held_y - model.predict(held_x.reshape(-1, 1))) ** 2).mean()))})

runaway = HistGradientBoostingRegressor(learning_rate=0.10, max_iter=1000,
                                        early_stopping=False, random_state=0).fit(
    fit_x.reshape(-1, 1), fit_y)
stopping_rows.append({
    "learning rate": 0.10, "early stopping": "NO", "iterations used": 1000,
    "held-out RMSE": float(np.sqrt(((held_y - runaway.predict(held_x.reshape(-1, 1))) ** 2).mean()))})

print(pd.DataFrame(stopping_rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.6))

full = GradientBoostingRegressor(n_estimators=600, learning_rate=0.05, max_depth=2,
                                 random_state=0).fit(fit_x.reshape(-1, 1), fit_y)
per_round = np.array([float(np.sqrt(((held_y - prediction) ** 2).mean()))
                      for prediction in full.staged_predict(held_x.reshape(-1, 1))])

ax.plot(np.arange(1, 601), per_round, color="#0072B2", linewidth=2.2,
        label="held-out RMSE, rate 0.05")
stopped_at = 105
ax.axvline(stopped_at, color="#009E73", linewidth=2.4,
           label="where early stopping halted (%d)" % stopped_at)
ax.plot([int(np.argmin(per_round)) + 1], [per_round.min()], "*", color="#D55E00",
        markersize=20, label="the true optimum (%d, %.4f)"
                             % (int(np.argmin(per_round)) + 1, per_round.min()))
ax.axhline(NOISE_SD, color="#000000", linestyle="--", linewidth=1.6,
           label="noise floor, %.2f" % NOISE_SD)
ax.set_ylim(1.45, 1.85)
ax.set_xlabel("boosting round")
ax.set_ylabel("held-out RMSE")
ax.set_title("Early stopping does not need to be exact - the curve is flat near its floor",
             fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Early stopping at rate 0.05 halts after 105 iterations at RMSE 1.5187 - essentially the best this data
allows. The same model run to 1,000 iterations without stopping reaches 1.6959.**

**Two details worth getting right.**

**`validation_fraction` carves the slice out of the training data**, not out of your test set. So the
model chooses its own stopping round using rows it then does not learn from - which means your test set
stays untouched, and 04-07's rule about not selecting on the data you report is respected.

**`n_iter_no_change` is a patience, not a threshold.** The validation error wobbles from round to round,
so stopping at the first increase would stop far too early. Twenty rounds of no improvement is a common
default; smaller is twitchy, larger wastes compute.

**A caveat on the numbers above:** the early-stopped models score slightly *worse* than the 1.5166 the
manual sweep found, because they trained on 20% fewer rows. That is the same "two things changed at once"
that 04-07 found in nested cross-validation - the honest price of letting the procedure be automatic.

## Boosting can be made robust, and that pays off 05-04

05-04 argued that MAE and MSE encode different beliefs about what a mistake costs, and that squared error
is dominated by a few bad rows. **That argument applies to the training loss, not just to the report** -
and boosting lets you change it.

### Predict before running

Five of the 200 training targets are corrupted by +40 - a data-entry error, a unit mix-up. What does that
do to a model trained on squared error?

In [ ]:
corrupted_y = fit_y.copy()
damaged = np.random.default_rng(3).choice(len(corrupted_y), 5, replace=False)
corrupted_y[damaged] += 40.0

robust_rows = []
for loss in ["squared_error", "absolute_error"]:
    model = GradientBoostingRegressor(loss=loss, n_estimators=200, learning_rate=0.05,
                                      max_depth=2, random_state=0).fit(
        fit_x.reshape(-1, 1), corrupted_y)
    robust_rows.append({
        "training loss": loss, "training data": "5 of 200 corrupted",
        "held-out RMSE": float(np.sqrt(((held_y - model.predict(held_x.reshape(-1, 1))) ** 2).mean()))})

clean = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2,
                                  random_state=0).fit(fit_x.reshape(-1, 1), fit_y)
robust_rows.append({"training loss": "squared_error", "training data": "clean",
                    "held-out RMSE": float(np.sqrt(((held_y - clean.predict(held_x.reshape(-1, 1))) ** 2).mean()))})
print(pd.DataFrame(robust_rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
grid = np.linspace(-4, 4, 400)

for ax, loss, title in [(left, "squared_error", "squared error: chases the five bad rows"),
                        (right, "absolute_error", "absolute error: ignores them")]:
    model = GradientBoostingRegressor(loss=loss, n_estimators=200, learning_rate=0.05,
                                      max_depth=2, random_state=0).fit(
        fit_x.reshape(-1, 1), corrupted_y)
    ax.scatter(fit_x, corrupted_y, s=16, alpha=0.4, color="#999999", label="training rows")
    ax.scatter(fit_x[damaged], corrupted_y[damaged], s=110, color="#D55E00",
               marker="X", zorder=4, label="the 5 corrupted rows")
    ax.plot(grid, true_curve(grid), color="#000000", linewidth=2.2, linestyle="--",
            label="the truth")
    ax.plot(grid, model.predict(grid.reshape(-1, 1)), color="#0072B2", linewidth=2.4,
            label="the fitted model")
    ax.set_xlabel("x")
    ax.set_title("%s\nheld-out RMSE %.4f"
                 % (title, np.sqrt(((held_y - model.predict(held_x.reshape(-1, 1))) ** 2).mean())),
                 fontsize=10.5)
    ax.legend(fontsize=8, loc="upper left")

left.set_ylabel("y")
plt.tight_layout()
plt.show()

> **Five corrupted rows out of two hundred - 2.5% of the data - take squared-error boosting from 1.5522
> to 5.4986. The same corruption with absolute-error boosting gives 1.6292.**

That is a factor of **3.4 in held-out RMSE**, from changing one argument.

**The mechanism is exactly 05-04's.** Squared error charges `(y - F)²`, so a residual of 40 contributes
1,600 - it dominates the gradient, and every round the trees chase it. Absolute error charges `|y - F|`,
whose gradient is the *sign* of the residual: a row that is 40 out pulls exactly as hard as a row that is
0.4 out, just in the same direction. **The corrupted rows get one vote each instead of sixteen hundred.**

**And the cost is visible too.** On this data the absolute-error model reaches 1.6292 where a
squared-error model on *clean* data reaches 1.5522, so robustness is not free - throwing away the
magnitude of the residual throws away real information along with the corrupted kind.

**When to reach for it:** when you suspect the target has recording errors, when the target has a genuine
long tail you do not want the model chasing, or when 05-05's residual plot shows a handful of enormous
residuals dominating everything. **Not by default** - on clean data squared error is better, and it is
better for a reason.

## Forest or boosting? The comparison, on the chapter that has followed us all module

In [ ]:
# SYNTHETIC: 05-07's and 05-10's 600 shop-days, with the promo x weekend interaction
shop_rng = np.random.default_rng(9)
n_days = 600
shop = pd.DataFrame({
    "footfall": shop_rng.uniform(50, 400, n_days),
    "promo": shop_rng.integers(0, 2, n_days).astype(float),
    "weekend": shop_rng.integers(0, 2, n_days).astype(float)})
sales = (200 + 0.9 * shop.footfall + 15 * shop.promo + 40 * shop.weekend
         + 120 * shop.promo * shop.weekend + shop_rng.normal(0, 25, n_days))

train_shop, test_shop, train_sales, test_sales = train_test_split(
    shop, sales, test_size=0.3, random_state=0)


def shop_rmse(model, X=test_shop, y=test_sales):
    return float(np.sqrt(((y - model.predict(X)) ** 2).mean()))


comparison = []
for label, model, told in [
        ("linear, main effects only", LinearRegression(), "no"),
        ("random forest of 300", RandomForestRegressor(n_estimators=300, random_state=0), "no"),
        ("boosting, 500 x depth 3, rate 0.05",
         GradientBoostingRegressor(n_estimators=500, max_depth=3, learning_rate=0.05,
                                   random_state=0), "no"),
        ("boosting, 200 x depth 2, rate 0.1",
         GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.1,
                                   random_state=0), "no")]:
    model.fit(train_shop, train_sales)
    comparison.append({"model": label, "given the interaction?": told,
                       "test RMSE": shop_rmse(model)})

with_product = train_shop.assign(px=train_shop.promo * train_shop.weekend)
told_model = LinearRegression().fit(with_product, train_sales)
comparison.append({"model": "linear, WITH promo x weekend", "given the interaction?": "YES",
                   "test RMSE": shop_rmse(told_model,
                                          test_shop.assign(px=test_shop.promo * test_shop.weekend))})

print(pd.DataFrame(comparison).to_string(index=False, float_format=lambda v: "%.3f" % v))
print("\nthe noise this data was built with has sd 25")

**Boosting at 200 trees of depth 2 reaches 25.250 - better than the forest's 29.375 and far better than
the main-effects linear model's 40.654 - and still does not catch the linear model that was handed the
interaction, at 23.412.**

**Three readings, in order of usefulness.**

**Boosting beat the forest by 4.1 RMSE on identical inputs.** Both find interactions; boosting spends its
capacity where the error still is, while the forest spends every tree on the whole problem. On tabular
data with structure to find, that difference is typical and it is why boosting dominates competitions.

**The deeper, longer boosted model is worse - 27.768 against 25.250.** More capacity, spent badly.
Depth 3 with 500 rounds has far more expressive power than depth 2 with 200 and uses it to fit noise:
the same lesson as 05-07's degree sweep, in a third parameterisation.

**And the linear model still wins**, because `footfall` enters the truth as a straight line and every
tree-based model has to build that line out of steps. 05-10 made this point about forests; boosting
narrows the gap without closing it.

> **The honest summary of this module: knowing the shape of the truth beats every algorithm here.** The
> best model on this data is the one where a human supplied a single product term. Boosting is what you
> reach for when nobody knows the shape - and it gets remarkably close.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.6))
ax.set_xlim(0, 10.6)
ax.set_ylim(0, 6.4)
ax.axis("off")

ax.text(5.3, 6.1, "FOREST versus BOOSTING", fontsize=13, fontweight="bold", ha="center")

ax.add_patch(plt.Rectangle((0.2, 3.2), 5.0, 2.6, facecolor="#DDEBF7",
                           edgecolor="#0072B2", linewidth=1.6))
ax.text(2.7, 5.45, "RANDOM FOREST", fontsize=12, fontweight="bold", ha="center")
for index, line in enumerate([
        "trees fitted INDEPENDENTLY, then averaged",
        "attacks VARIANCE",
        "more trees is never worse",
        "almost no tuning needed",
        "trivially parallel",
        "the strong no-effort baseline"]):
    ax.text(0.35, 5.05 - index * 0.33, line, fontsize=9.5)

ax.add_patch(plt.Rectangle((5.4, 3.2), 5.0, 2.6, facecolor="#D9EAD3",
                           edgecolor="#009E73", linewidth=1.6))
ax.text(7.9, 5.45, "GRADIENT BOOSTING", fontsize=12, fontweight="bold", ha="center")
for index, line in enumerate([
        "trees fitted IN SEQUENCE, on the errors",
        "attacks BIAS",
        "more trees WILL overfit - stop early",
        "learning rate and depth matter",
        "sequential by construction",
        "usually the better model, with effort"]):
    ax.text(5.55, 5.05 - index * 0.33, line, fontsize=9.5)

ax.text(0.2, 2.65, "THE RECIPE THAT USUALLY WORKS", fontsize=11, fontweight="bold")
for index, line in enumerate([
        "small learning rate (0.05 or 0.1) + many rounds + early stopping on a validation slice",
        "shallow trees (depth 2-4) - the ensemble supplies the capacity, not each tree",
        "absolute_error loss if the target has recording errors: 5.4986 became 1.6292 here"]):
    ax.text(0.35, 2.25 - index * 0.38, "* " + line, fontsize=9.5)

ax.text(0.2, 0.95, "AND WHAT NEITHER CAN DO", fontsize=11, fontweight="bold",
        color="#CC0000")
ax.text(0.35, 0.55, "Extrapolate. Both are still made of trees, and 05-10's flat line past the",
        fontsize=9.8)
ax.text(0.35, 0.15, "edge of the training range applies unchanged to every model on this page.",
        fontsize=9.8)

plt.tight_layout()
plt.show()

## Common misconceptions

**"Boosting is just a better random forest."**
Different mechanism and a different target. A forest averages independent trees to reduce **variance**; a
boosted model builds dependent trees to reduce **bias**. The consequence is the one that catches people:
adding trees is free in a forest and dangerous in boosting.

**"More rounds is more accurate."**
Held-out RMSE bottomed at round 10 in this chapter's first example and rose from 1.5179 to 1.6097 by
round 60. The number of rounds is a capacity dial with an optimum.

**"A smaller learning rate gives a better model."**
It gives the *same* model, later, with more forgiveness about when you stop. Rates from 1.0 to 0.01 all
reached 1.52 to 1.61 at their own best round; what changed was the round (8 against 323) and the cost of
overshooting (0.46 against 0.0033).

**"Boosting needs its features scaled."**
It is made of trees. No.

**"Early stopping is optional."**
It is the mechanism that sets the one hyperparameter that cannot be left at a default. Without it, the
same model that scored 1.5187 scored 1.6959.

**"Gradient boosting is a black box."**
The loop is eight lines and reproduces the library to four decimals. What is hard to interpret is a
thousand trees, which is a different complaint - and 05-10's warnings about feature importance apply
here too, with the additional wrinkle that later trees are fitted to residuals rather than to the target.

**"XGBoost / LightGBM / CatBoost are different algorithms."**
They are the same algorithm with better engineering - histogram binning, smarter split-finding, native
categorical handling, regularisation on the leaf values. `HistGradientBoostingRegressor` in scikit-learn
is the same family. **Learn the loop above and you have learned all of them.**

## Exercises

Solutions: `solutions/05_regression/05-11_boosting_solutions.ipynb`.

### Quick understanding

**E1.** In one sentence each, say what a forest and a boosted model each do with their trees, and which
term of the decomposition each attacks.

**E2.** Why is "more trees" safe in a forest and dangerous in boosting?

**E3.** Why is fitting a tree to the residuals a gradient step?

### Hand calculation

**E4.** Targets 10, 14, 20. The first model predicts the mean. Give the residuals, and the new predictions
after adding 0.5 times a "tree" that predicts each residual exactly.

**E5.** Continue E4 for two more rounds. What are the predictions converging to, and how fast?

**E6.** At a learning rate of 0.1, how many rounds of the E4 process are needed before the largest
remaining residual is below 0.1 of its starting size?

**E7.** A boosted model has training RMSE 0.4 and held-out RMSE 2.2 after 800 rounds; at round 60 they
were 1.8 and 1.9. What happened, and what is the model you should ship?

### Coding

**E8.** Extend this chapter's hand-written loop to accept any `max_depth` and any learning rate, and
verify it against `GradientBoostingRegressor` for three different settings.

**E9.** Add subsampling to your loop - fit each tree on a random 60% of the rows - and report what it does
to the held-out curve. This is "stochastic gradient boosting"; explain the name.

**E10.** Use `staged_predict` to find the best round for depths 1, 2, 3 and 5 at a fixed learning rate.
Report the best round and best RMSE for each, and explain the pattern in the best rounds.

**E11.** Compare `GradientBoostingRegressor` and `HistGradientBoostingRegressor` on 50,000 synthetic rows:
fit time and held-out score. Explain the difference in one sentence.

**E12.** Implement early stopping yourself - a validation slice, a patience, and a record of the best
model - and check it lands within one round of the `staged_predict` optimum.

### Interpretation

**E13.** Your boosted model's validation curve is still falling at 5,000 rounds. What does that tell you,
and what would you change?

**E14.** Boosting beats your linear model by 30% and your forest by 2%. What do you conclude about the
data, and what would you try next?

### Debugging

**E15.** A boosted model scores brilliantly in cross-validation and poorly in production. Give three
causes, at least two of which are specific to boosting or to trees.

**E16.** Your boosted model takes four hours to train. Name five things to try, in the order you would try
them.

### Exam and interview reasoning

**E17.** "What is the difference between bagging and boosting?" Answer in under a minute, then handle:
"which would you use on 500 rows, and why?"

### Transfer to a different situation

**E18.** You are asked to predict insurance claim amounts: most claims are zero or small, a few are
enormous, and the business cares most about not underestimating the large ones. Say what you would do
about the loss, the model and the evaluation.

### Explain it to someone non-technical

**E19.** In under 90 words, explain boosting, and why it can be made worse by trying harder.

### Optional challenge

**E20.** Implement boosting with absolute-error loss - fit each tree to the *sign* of the residual - and
compare it with `loss="absolute_error"` on data with corrupted targets.

**E21.** Show that boosting with a linear model as the base learner, at learning rate 1, reproduces
ordinary least squares in a single round - and explain what that says about why the base learner must be
weak.

## Mastery check

- [ ] Write the boosting loop from memory and say what each line does
- [ ] Explain the gradient connection without hand-waving
- [ ] Read a staged validation curve and pick the stopping round
- [ ] Say what a smaller learning rate buys and what it costs
- [ ] Choose between a forest and a boosted model, with a reason
- [ ] Recognise when a robust loss is worth its price

## What should now feel instinctive

- Setting a small learning rate and letting early stopping choose the rounds
- Treating "number of rounds" as capacity, not as effort
- Reaching for boosting when a forest is good and you need better
- Checking the residual distribution before accepting squared error as the training loss
- Remembering that every model in this chapter is still a tree, and still cannot extrapolate

## Flashcards

| Front | Back |
|---|---|
| The boosting loop | start at the mean; fit a tree to the residuals; add `rate x` its prediction; repeat |
| Why it is called gradient boosting | the negative gradient of squared error w.r.t. the prediction *is* the residual |
| Forest vs boosting | independent trees averaged (variance) against sequential trees on the errors (bias) |
| More trees | free in a forest; overfits in boosting - 1.5179 at round 10, 1.6097 at round 60 |
| What the learning rate buys | not a better optimum; a later one, and forgiveness about missing it |
| Rate 1.0 against rate 0.01 | best at round 8 against round 323; overshooting costs 0.46 against 0.0033 |
| Early stopping | 105 iterations at 1.5187, against 1,000 iterations at 1.6959 |
| A robust loss | 5 of 200 rows corrupted: squared 5.4986, absolute 1.6292 |
| On the shop data | boosting 25.250, forest 29.375, linear with the interaction 23.412 |
| XGBoost, LightGBM, CatBoost | the same algorithm, better engineered |
| What boosting still cannot do | extrapolate. It is made of trees |

## Next

**05-12 · The applied checkpoint.** Eleven chapters of pieces - a framing, a baseline, a split, a metric,
residuals, capacity, regularisation, trees, boosting - assembled into one honest end-to-end regression on
a real dataset, with every decision recorded and defended.

No new algorithms. The exercise is doing the whole thing once, in order, without skipping the steps that
are boring - which is the difference between knowing these chapters and being able to use them.